# 01 — Análisis Exploratorio de Datos (EDA)

**Proyecto Big Data — NYC Taxi · Spark MLlib**

EDA escalable de los viajes de taxi de NYC: estructura, calidad, desbalance de clases y patrones temporales. Las agregaciones se hacen en Spark (escalables a todo el volumen) y solo los resúmenes pequeños se llevan a pandas para graficar.

In [ ]:
import sys; sys.path.insert(0, '../src')
from nyc_taxi_mllib import build_spark, load_trips, clean_trips, add_label, class_balance
from pyspark.sql import functions as F
spark = build_spark('eda-notebook')
spark.version

## 1. Ingesta
Lectura de los Parquet mensuales (esquema normalizado a tipos canónicos).

In [ ]:
raw = load_trips(spark)
print('Viajes crudos:', raw.count())
raw.printSchema()

## 2. Limpieza y etiqueta
Filtrado de inválidos, solo pagos con tarjeta (veracidad de la propina), variables temporales/cinemáticas y etiqueta de *propina baja* (<10%).

In [ ]:
df = add_label(clean_trips(raw)).cache()
print('Viajes limpios:', df.count())
df.select('trip_distance','trip_duration_min','fare_amount','tip_amount','label').show(5)

## 3. Desbalance de clases
La clase positiva (propina baja) es minoritaria: el reto central del proyecto.

In [ ]:
bal = class_balance(df)
print(bal)
print(f"Propina baja: {bal['pos_rate']:.2%}  |  Desbalance {bal['imbalance_ratio']:.1f}:1")

## 4. Patrón temporal
Tasa de propina baja por hora del día.

In [ ]:
(df.groupBy(F.hour('tpep_pickup_datetime').alias('hora'))
   .agg(F.mean('label').alias('tasa_propina_baja'))
   .orderBy('hora')).show(24)

## 5. Figuras
El script `scripts/run_eda.py` genera todas las figuras publication-ready (paleta Okabe-Ito) en `figures/`. Aquí se invoca el mismo pipeline.

In [ ]:
# Genera figuras fig1..fig6 en ../figures/
import subprocess; print(subprocess.run([sys.executable, '../scripts/run_eda.py'],
      capture_output=True, text=True).stdout[-400:])

In [ ]:
spark.stop()